<div style="padding:25px;background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);color:white;border-radius:12px;font-family:'Segoe UI',Arial,sans-serif;"><h1 style="color:white;margin:0;font-size:2.2em;">Cognitive Knowledge Assistant</h1><p style="margin:8px 0 0 0;opacity:.9;">AI-Powered RAG Assistant with PDF &amp; Sheet Integration and Chat Memory</p><div style="margin-top:15px;border-top:1px solid rgba(255,255,255,.2);padding-top:10px;font-size:.9em;">📚 Mid-term Project &nbsp;|&nbsp; ⚡ LangChain · FAISS · Groq · Gemini Embeddings</div></div>

### 🛠️ Step 1: Install Required Libraries
> ⚠️ After this cell finishes → **Runtime → Restart session** → then run from Step 2.

In [ ]:
!pip install -q -U google-generativeai
!pip install -q langchain langchain-community langchain-groq langchain-text-splitters faiss-cpu pypdf pandas openpyxl


### 📦 Step 2: Import Libraries
We use `google-generativeai` for embeddings — no local model download, no PyTorch.

In [ ]:
import os
import re
import getpass
import pandas as pd
from typing import List
from google.colab import files

import google.generativeai as genai
from langchain_core.embeddings import Embeddings

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_groq import ChatGroq

try:
    from langchain_classic.memory import ConversationBufferMemory
    from langchain_classic.chains import ConversationalRetrievalChain
except ImportError:
    from langchain.memory import ConversationBufferMemory
    from langchain.chains import ConversationalRetrievalChain

print("All imports successful!")


### 🔑 Step 3: Enter API Keys
- **Google AI key**: from [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
- **Groq key**: from [console.groq.com/keys](https://console.groq.com/keys)

In [ ]:
google_api_key = getpass.getpass("Enter your Google AI API key: ")
os.environ["GOOGLE_API_KEY"] = google_api_key
genai.configure(api_key=google_api_key)

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
print("Keys set!")


### 🧬 Step 4: Define Embedding Model
We wrap the Gemini embedding API in a LangChain-compatible class. We use models/gemini-embedding-001 (the standard replacement for the deprecated text-embedding-004 model).

In [ ]:
class GeminiEmbeddings(Embeddings):
    """LangChain-compatible wrapper for Google Gemini gemini-embedding-001."""

    def __init__(self, model: str = "models/gemini-embedding-001"):
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        embeddings = []
        for text in texts:
            result = genai.embed_content(
                model=self.model,
                content=text,
                task_type="retrieval_document"
            )
            embeddings.append(result["embedding"])
        return embeddings

    def embed_query(self, text: str) -> List[float]:
        result = genai.embed_content(
            model=self.model,
            content=text,
            task_type="retrieval_query"
        )
        return result["embedding"]

embedding_model = GeminiEmbeddings()
print("Embedding model ready!")


### 📂 Step 5: Ingest Data Sources
Upload a PDF and optionally add a Google Sheet link.

In [ ]:
uploaded_pdf = files.upload()
pdf_filename = list(uploaded_pdf.keys())[0]

all_documents = []

pdf_loader = PyPDFLoader(pdf_filename)
pdf_docs = pdf_loader.load()
for doc in pdf_docs:
    doc.metadata["source"] = pdf_filename
all_documents.extend(pdf_docs)
print(f"Pages Loaded from PDF: {len(pdf_docs)}")

google_sheet_url = input("Enter Google Sheet Link (press Enter to skip): ").strip()

if google_sheet_url:
    match = re.search(r"/d/([a-zA-Z0-9-_]+)", google_sheet_url)
    if match:
        sheet_id = match.group(1)
        csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"
        try:
            df = pd.read_csv(csv_url)
            sheet_docs = []
            for idx, row in df.iterrows():
                row_text = " | ".join([f"{col}: {val}" for col, val in row.items() if pd.notna(val)])
                sheet_docs.append(Document(
                    page_content=row_text,
                    metadata={"source": "Google Sheet", "row": idx}
                ))
            all_documents.extend(sheet_docs)
            print(f"Rows Loaded from Google Sheet: {len(sheet_docs)}")
        except Exception as e:
            print(f"Could not load sheet: {e}")


### ✂️ Step 6: Split Text into Chunks
Documents are split into overlapping passages to preserve local context.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(all_documents)
print(f"Chunks Created: {len(split_docs)}")


### 🗄️ Step 7: Build Vector Database
FAISS indexes all chunks for fast semantic similarity search.

In [ ]:
vector_store = FAISS.from_documents(split_docs, embedding_model)
print("FAISS index created!")


### 🤖 Step 8: Load Language Model
We use Llama-3.3 70B via Groq for fast, accurate answer generation.

In [ ]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0
)
print("LLM loaded!")


### 🧠 Step 9: Initialize Conversation Memory
ConversationBufferMemory stores the full chat history to maintain context across turns.

In [ ]:
chat_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)
print("Memory initialized!")


### 🔗 Step 10: Build Conversational RAG Chain
We combine the LLM, retriever, and memory into a single conversational pipeline.

In [ ]:
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
    memory=chat_memory,
    return_source_documents=True
)

print("RAG Chain ready!")


### 💬 Step 11: Interactive Chat
Type your questions below. Use `exit` to quit or `clear` to reset memory.

In [ ]:
while True:
    question = input("You: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    if question.lower() == "clear":
        chat_memory.clear()
        print("Memory cleared.")
        continue

    response = qa_chain.invoke({"question": question})

    print("\nAssistant:")
    print(response["answer"])
    print("\n[Sources]:")
    for idx, doc in enumerate(response["source_documents"]):
        src = doc.metadata.get("source", "Unknown")
        loc = doc.metadata.get("row", doc.metadata.get("page", "N/A"))
        print(f"  [{idx+1}] {src}  (loc: {loc})")
    print("-" * 60)
